# Notebook 4: calibration

Calibration diagnostics for compatible saved probabilistic forecasts.

In [ ]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import xlogy

selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
countries = load_data(network="geographic")["countries"]

core = load_result("nb1_primary_models")
bench = load_result("nb2_benchmarks")

CANDIDATES = dict(core["forecast_bundles"])
for name, bundle in bench["forecast_bundles"].items():
    if bundle is not None:
        CANDIDATES[name] = bundle

loaded = {}
for name, bundle in CANDIDATES.items():
    try:
        loaded[name] = load_forecasts(bundle)
    except FileNotFoundError:
        print(f"forecast bundle unavailable: {bundle}")

print(f"loaded {len(loaded)} forecast bundles")

In [ ]:
def pit(y, mu, sd):
    """Probability integral transform under a Gaussian predictive."""
    return stats.norm.cdf((y - mu) / np.maximum(sd, 1e-8))

def kupiec_lr(hits, alpha=0.05):
    """Kupiec unconditional-coverage likelihood-ratio test."""
    hits = np.asarray(hits, dtype=int)
    n = len(hits)
    if n == 0:
        return np.nan, np.nan
    x = int(hits.sum())
    pi = x / n
    ll_null = xlogy(n - x, 1 - alpha) + xlogy(x, alpha)
    ll_alt = xlogy(n - x, 1 - pi) + xlogy(x, pi)
    lr = float(-2 * (ll_null - ll_alt))
    return lr, float(stats.chi2.sf(lr, 1))

def christoffersen_lr(hits):
    """Christoffersen independence test for interval exceedances."""
    h = np.asarray(hits, dtype=int)
    if len(h) < 2:
        return np.nan, np.nan
    n00 = int(np.sum((h[:-1] == 0) & (h[1:] == 0)))
    n01 = int(np.sum((h[:-1] == 0) & (h[1:] == 1)))
    n10 = int(np.sum((h[:-1] == 1) & (h[1:] == 0)))
    n11 = int(np.sum((h[:-1] == 1) & (h[1:] == 1)))
    d0, d1 = n00 + n01, n10 + n11
    if d0 == 0 or d1 == 0:
        return np.nan, np.nan
    p01, p11 = n01 / d0, n11 / d1
    pooled = (n01 + n11) / (d0 + d1)
    ll_null = xlogy(n00 + n10, 1 - pooled) + xlogy(n01 + n11, pooled)
    ll_alt = (xlogy(n00, 1 - p01) + xlogy(n01, p01)
              + xlogy(n10, 1 - p11) + xlogy(n11, p11))
    lr = float(-2 * (ll_null - ll_alt))
    return lr, float(stats.chi2.sf(lr, 1))

def assess(fc, alpha=0.05):
    """Return country-level and panel calibration summaries."""
    y, mu = fc["y_true"], fc["mean"]
    sd = np.sqrt(np.maximum(fc["var"], 1e-12))
    if y.shape[1] != len(countries):
        raise ValueError("Calibration bundle does not contain the full country panel.")

    rows = []
    pit_values = {}
    for i, country in enumerate(countries):
        u = pit(y[:, i], mu[:, i], sd[:, i])
        pit_values[country] = u.tolist()
        ks = stats.kstest(u, "uniform")
        cvm = stats.cramervonmises(u, "uniform")
        hits = ((y[:, i] < mu[:, i] - 1.96 * sd[:, i])
                | (y[:, i] > mu[:, i] + 1.96 * sd[:, i]))
        lr_k, p_k = kupiec_lr(hits)
        lr_c, p_c = christoffersen_lr(hits)
        rows.append({
            "country": country,
            "coverage_95": float(1 - hits.mean()),
            "interval_width": float(np.mean(2 * 1.96 * sd[:, i])),
            "KS_stat": float(ks.statistic), "KS_p": float(ks.pvalue),
            "CvM_stat": float(cvm.statistic), "CvM_p": float(cvm.pvalue),
            "Kupiec_LR": lr_k, "Kupiec_p": p_k,
            "Christoffersen_LR": lr_c, "Christoffersen_p": p_c,
        })

    per = pd.DataFrame(rows)
    fractions, n_defined = {}, {}
    for label, col in [
        ("KS", "KS_p"), ("CvM", "CvM_p"), ("Kupiec", "Kupiec_p"),
        ("Christoffersen", "Christoffersen_p"),
    ]:
        valid = per[col].notna()
        n_defined[label] = int(valid.sum())
        fractions[label] = float((per.loc[valid, col] < alpha).mean()) if valid.any() else np.nan
    return fractions, n_defined, per, pit_values

full, defined, per_country, pit_values = {}, {}, {}, {}
for name, fc in loaded.items():
    full[name], defined[name], per_country[name], pit_values[name] = assess(fc)

print("Rejection fractions at 5%:")
print(pd.DataFrame(full).T.round(5).to_string())

In [ ]:
save_result("nb4_calibration", {
    "full_sample": full,
    "n_defined": defined,
    "per_country": {k: v.to_dict(orient="records") for k, v in per_country.items()},
    "pit_values": pit_values,
    "forecast_bundles": CANDIDATES,
    "config": run_config(p=P, stages=[2] * P, purpose="forecast_calibration"),
})
print("saved nb4_calibration")